# Tournament Retrieval

Code here adopted from Emilio Rivera: https://github.com/ERiverIllini/smash_misc/releases

In [11]:
import requests
import json
import pandas as pd
from keys import api_key
import os
from datetime import datetime

In [12]:
AUTH_TOKEN = api_key
SF_BASED_COORDS = "37.77151615492457, -122.41563048985462"
SF_RADIUS = "70mi"

SAC_BASED_COORDS = "38.57608096237729, -121.49183616631059"
SAC_RADIUS = "40mi"

NUM_PER_PAGE = 50
request_url = 'https://api.start.gg/gql/alpha'

In [13]:
def flatten_nested_json_df(df):
    
    df = df.reset_index()
    
    print(f"original shape: {df.shape}")
    print(f"original columns: {df.columns}")
    
    s = (df.applymap(type) == list).all()
    list_columns = s[s].index.tolist()
    
    s = (df.applymap(type) == dict).all()
    dict_columns = s[s].index.tolist()
    
    print(f"lists: {list_columns}, dicts: {dict_columns}")
    while len(list_columns) > 0 or len(dict_columns) > 0:
        new_columns = []
        
        for col in dict_columns:
            print(f"flattening: {col}")
            horiz_exploded = pd.json_normalize(df[col]).add_prefix(f'{col}.')
            horiz_exploded.index = df.index
            df = pd.concat([df, horiz_exploded], axis=1).drop(columns=[col])
            new_columns.extend(horiz_exploded.columns) 
        
        for col in list_columns:
            print(f"exploding: {col}")
            df = df.drop(columns=[col]).join(df[col].explode().to_frame())
            df = df.reset_index(drop=True)
            new_columns.append(col)
        
        s = (df[new_columns].applymap(type) == list).all()
        list_columns = s[s].index.tolist()

        s = (df[new_columns].applymap(type) == dict).all()
        dict_columns = s[s].index.tolist()
        
        print(f"lists: {list_columns}, dicts: {dict_columns}")
        
    print(f"final shape: {df.shape}")
    print(f"final columns: {df.columns}")
    return df

In [14]:
def date_to_unix(date_string, date_format="%Y-%m-%d"):
    try:
        date_obj = datetime.strptime(date_string, date_format)
        unix_timestamp = int(date_obj.timestamp())
        return unix_timestamp
    except ValueError as e:
        print(f"Error: {e}")
        return None

In [15]:
def generate_graphql_query(timestamps):
    template = """
query BayNorCalTournaments($page: Int, $perPage: Int, $coordinates: String!, $radius: String!) {{
  tournaments(
    query: {{
    page: $page
    perPage: $perPage
    filter: {{
      location: {{
        distanceFrom: $coordinates,
        distance: $radius
      }},
      afterDate: {after_date} 
      beforeDate: {before_date}
    }}
    sortBy:"startAt"
  }}) {{
    nodes {{
      id
      name
      city
      slug
      startAt
      events {{
        slug
        numEntrants
        videogame {{
          name
        }}
      }}
    }}
  }}
}}
"""
    after_date = timestamps[0]
    before_date = timestamps[1]
    query = template.format(after_date=after_date, before_date=before_date)

    return query

In [16]:
def get_all_tournies(auth_token, query, coords, radius, num_per_page):
  
  graphql_query = query 
  tournies = []

  for i in range(1, 10):
    variables = {
        "page": i,
        "perPage": num_per_page,
        "coordinates": coords,
        "radius": "50mi"
    }
    data = {"query" : graphql_query, "variables": variables}
    json_data = json.dumps(data)
    auth_header = auth_token
    header = {'Authorization': 'Bearer ' + auth_header}  

    response = requests.post(url=request_url, headers=header, data=json_data)
    json_resp = json.loads(response.text)
    print(json_resp)
    if ("errors" not in json_resp):
       
      curr_tournies_page = json_resp['data']['tournaments']['nodes']
      print("Number of tournies in page is:" + str(len(curr_tournies_page)))

      tournies += curr_tournies_page
  return tournies

In [17]:
start_date = "2024-10-01"
end_date = "2024-12-31"
timestamps = [date_to_unix(start_date), date_to_unix(end_date)]
timestamps

[1727766000, 1735632000]

In [18]:
query = generate_graphql_query(timestamps)
print(query)


query BayNorCalTournaments($page: Int, $perPage: Int, $coordinates: String!, $radius: String!) {
  tournaments(
    query: {
    page: $page
    perPage: $perPage
    filter: {
      location: {
        distanceFrom: $coordinates,
        distance: $radius
      },
      afterDate: 1727766000 
      beforeDate: 1735632000
    }
    sortBy:"startAt"
  }) {
    nodes {
      id
      name
      city
      slug
      startAt
      events {
        slug
        numEntrants
        videogame {
          name
        }
      }
    }
  }
}



In [21]:
tournies = []
bay_tournies = get_all_tournies(AUTH_TOKEN, query, SF_BASED_COORDS, SF_RADIUS, NUM_PER_PAGE)
sac_tournies = get_all_tournies(AUTH_TOKEN, query, SAC_BASED_COORDS, SAC_RADIUS, NUM_PER_PAGE)
tournies = sac_tournies + bay_tournies
np_tournies = pd.DataFrame(tournies).explode('events')
flat_tournies = flatten_nested_json_df(np_tournies)

{'data': {'tournaments': {'nodes': [{'id': 716230, 'name': 'Tech City Tekken #146 - Tekken 8', 'city': 'Sunnyvale', 'slug': 'tournament/tech-city-tekken-146-tekken-8', 'startAt': 1728009000, 'events': [{'slug': 'tournament/tech-city-tekken-146-tekken-8/event/tekken-8-singles', 'numEntrants': 22, 'videogame': {'name': 'TEKKEN 8'}}]}, {'id': 714689, 'name': 'Slugfest #121 - SF6 Weekly', 'city': 'Sunnyvale', 'slug': 'tournament/slugfest-121-sf6-weekly-1', 'startAt': 1727928000, 'events': [{'slug': 'tournament/slugfest-121-sf6-weekly-1/event/street-fighter-6-singles', 'numEntrants': 13, 'videogame': {'name': 'Street Fighter 6'}}]}, {'id': 715107, 'name': 'Smash the Galaxy #145', 'city': 'Dublin', 'slug': 'tournament/smash-the-galaxy-145', 'startAt': 1728252000, 'events': [{'slug': 'tournament/smash-the-galaxy-145/event/street-fighter-6', 'numEntrants': 16, 'videogame': {'name': 'Street Fighter 6'}}, {'slug': 'tournament/smash-the-galaxy-145/event/mk1', 'numEntrants': 7, 'videogame': {'name

/var/folders/67/zc39wmb54bx3kc4n_tyb3g3m0000gn/T/ipykernel_78862/1626888527.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  s = (df.applymap(type) == list).all()
/var/folders/67/zc39wmb54bx3kc4n_tyb3g3m0000gn/T/ipykernel_78862/1626888527.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  s = (df.applymap(type) == dict).all()


In [22]:
ult_tournies = flat_tournies[
    flat_tournies['events'].apply(
        lambda x: isinstance(x, dict) and x.get('videogame', {}).get('name') == 'Super Smash Bros. Ultimate'
    ) &
    flat_tournies['events'].apply(
        lambda x: isinstance(x, dict) and x.get('numEntrants', 0) >= 16
    )
]

ult_tournies = ult_tournies.reset_index(drop=True)

In [23]:
ult_tournies['startgg_url'] = ult_tournies['events'].apply(lambda x: x['slug'] if isinstance(x, dict) and 'slug' in x else None)
ult_tournies['startgg_url'] = 'start.gg/' + ult_tournies['startgg_url'].astype('str')
ult_tournies['Event Date'] = ult_tournies['startAt'].map(
    lambda x: datetime.fromtimestamp(x).strftime('%Y-%m-%d') if pd.notnull(x) and isinstance(x, (int, float)) else None
)
ult_tournies['StartGG TOURNAMENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-3:-2][0])
ult_tournies['StartGG EVENT_ID'] = ult_tournies['startgg_url'].map(lambda url: url.split("/")[-1:][0]) 
ult_tournies = ult_tournies.drop_duplicates('startgg_url', keep='first')

ult_tournies['jakeSlug'] = ult_tournies['slug'] + '/event/' + ult_tournies['StartGG EVENT_ID']
ult_tournies #can export this to csv to use with braacket

,index,id,name,city,slug,startAt,events,startgg_url,Event Date,StartGG TOURNAMENT_ID,StartGG EVENT_ID,jakeSlug
0,0,715326,Cornerstone #56,Sacramento,tournament/cornerstone-56,1727920800,{'slug': 'tournament/cornerstone-56/event/ulti...,start.gg/tournament/cornerstone-56/event/ultim...,2024-10-02,cornerstone-56,ultimate-singles,tournament/cornerstone-56/event/ultimate-singles
1,1,720016,Cornerstone #57,Sacramento,tournament/cornerstone-57,1729130400,{'slug': 'tournament/cornerstone-57/event/ulti...,start.gg/tournament/cornerstone-57/event/ultim...,2024-10-16,cornerstone-57,ultimate-singles,tournament/cornerstone-57/event/ultimate-singles
2,4,728005,CECC West Open Tournaments,Sacramento,tournament/cecc-west-open-tournaments,1731780000,{'slug': 'tournament/cecc-west-open-tournament...,start.gg/tournament/cecc-west-open-tournaments...,2024-11-16,cecc-west-open-tournaments,ssbu-high-school-and-under-bracket-saturday,tournament/cecc-west-open-tournaments/event/ss...
3,4,728005,CECC West Open Tournaments,Sacramento,tournament/cecc-west-open-tournaments,1731780000,{'slug': 'tournament/cecc-west-open-tournament...,start.gg/tournament/cecc-west-open-tournaments...,2024-11-16,cecc-west-open-tournaments,ssbu-open-bracket-sunday,tournament/cecc-west-open-tournaments/event/ss...
4,5,732505,Dairy Free Series at UC Davis#6,Davis,tournament/dairy-free-series-at-uc-davis-6,1732388400,{'slug': 'tournament/dairy-free-series-at-uc-d...,start.gg/tournament/dairy-free-series-at-uc-da...,2024-11-23,dairy-free-series-at-uc-davis-6,ultimate-singles,tournament/dairy-free-series-at-uc-davis-6/eve...
...,...,...,...,...,...,...,...,...,...,...,...,...
61,234,705610,The Throne 2,San Jose,tournament/the-throne-2,1731096000,{'slug': 'tournament/the-throne-2/event/ultima...,start.gg/tournament/the-throne-2/event/ultimat...,2024-11-08,the-throne-2,ultimate-singles,tournament/the-throne-2/event/ultimate-singles
62,240,738023,Berkeley Ultimate Smashfest #27,Berkeley,tournament/berkeley-ultimate-smashfest-27,1733608800,{'slug': 'tournament/berkeley-ultimate-smashfe...,start.gg/tournament/berkeley-ultimate-smashfes...,2024-12-07,berkeley-ultimate-smashfest-27,singles,tournament/berkeley-ultimate-smashfest-27/even...
63,241,738587,Guildhouse Weekly 159 - SSBU & GGST,San Jose,tournament/guildhouse-weekly-159-ssbu-ggst,1733970600,{'slug': 'tournament/guildhouse-weekly-159-ssb...,start.gg/tournament/guildhouse-weekly-159-ssbu...,2024-12-11,guildhouse-weekly-159-ssbu-ggst,smash-ultimate-singles-switch-7-30-pm,tournament/guildhouse-weekly-159-ssbu-ggst/eve...
64,242,739402,Finals Destination 15,Berkeley,tournament/finals-destination-15,1734206400,{'slug': 'tournament/finals-destination-15/eve...,start.gg/tournament/finals-destination-15/even...,2024-12-14,finals-destination-15,ultimate-singles,tournament/finals-destination-15/event/ultimat...


# ELO Generation
- One issue that needs to be addressed here is that not all of these tournaments may necessarily count for PR as it scrapes 16+ attendance SSBU tournaments -- does not account for tournaments with banned players in attendance, free entry/no pot, or tournaments where the TO used Smash Ultimate as the tournament type but did not operate under a competitive ruleset

- From here we use Jake's code